<a href="https://colab.research.google.com/github/anshumanshukl1605/question-practice-/blob/master/Session_2_3_Multiple_Regression_Overfitting_Ridge_and_Lasso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2.3: Multiple Regression, Overfitting, Ridge and Lasso

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
!gdown 1jXcemrjeqSYLx16RS0qNBA6rokDcxvne

Downloading...
From: https://drive.google.com/uc?id=1jXcemrjeqSYLx16RS0qNBA6rokDcxvne
To: /content/house_listing.csv
100% 7.60k/7.60k [00:00<00:00, 21.0MB/s]


In [ ]:
listings = pd.read_csv('house_listing.csv')
listings.shape

(120, 13)

In [ ]:
listings.head()

,area_sqft,bedrooms,age_years,distance_km,extra_1,extra_2,extra_3,extra_4,extra_5,extra_6,extra_7,extra_8,price_lakhs
0,1860,5,0,6.0,1.06,0.89,0.22,-0.81,-0.99,1.21,-0.38,1.64,134.1
1,440,4,35,4.7,-0.49,-1.06,-0.24,1.31,0.16,-1.02,0.06,-0.67,57.7
2,996,3,25,20.7,-1.48,-1.70,0.22,0.10,-0.45,-1.24,1.11,-1.38,56.3
3,1630,5,2,19.6,-0.72,-0.65,0.17,-0.95,-0.50,-0.18,0.51,-0.25,114.0
4,455,1,4,15.8,-1.11,-0.01,-0.18,-0.32,0.40,-0.95,-0.48,0.84,37.8


In [ ]:
core = ["area_sqft", "bedrooms", "age_years", "distance_km"]


X = listings.drop(columns="price_lakhs")      # all 12 feature columns
y = listings["price_lakhs"]

In [ ]:
X.shape

(120, 12)

In [ ]:
y.shape

(120,)

In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)

print("train:", X_tr.shape, "| validation:", X_val.shape, "| test:", X_test.shape)

train: (72, 12) | validation: (24, 12) | test: (24, 12)


In [ ]:
model = LinearRegression()
model.fit(X_tr[core], y_tr) # my model is trained only on 4 features.

LinearRegression()

In [ ]:
print(pd.Series(model.coef_, index=core).round(3))
print("intercept:", round(model.intercept_, 2))

area_sqft      0.050
bedrooms       4.547
age_years     -0.410
distance_km   -1.484
dtype: float64
intercept: 35.39


In [ ]:
new_house = [[1000, 3, 10, 8]]
print("Predicted price:", round(model.predict(new_house)[0], 1), "lakhs")

Predicted price: 82.8 lakhs


In [ ]:
X_val[core].shape

(24, 4)

In [ ]:
pred_val = model.predict(X_val[core])
pred_val

array([ 93.99097798,  84.63789917,  92.61991259,  77.64056959,
        95.82978503,  24.00029093,  90.88418255,  78.43107023,
        85.69937568,  57.04640899, 103.43411715,  84.05750215,
        81.22656792, 105.09192152,  43.76929257,  99.87321772,
        96.8538647 ,  70.39125124,  88.59095831, 103.53751316,
        45.82835866, 102.73128514, 108.23143578, 146.75570404])

In [ ]:
mean_absolute_error(y_val, pred_val)

6.609900308227743

In [ ]:
root_mean_squared_error(y_val, pred_val)

8.27994744668094

In [ ]:
print("MAE: ", round(mean_absolute_error(y_val, pred_val), 2), "lakhs")
print("RMSE:", round(root_mean_squared_error(y_val, pred_val), 2), "lakhs")


MAE:  6.61 lakhs
RMSE: 8.28 lakhs


# Overfitting

In [ ]:
def show(name, model, X_train, X_valid):
    train_rmse = root_mean_squared_error(y_tr, model.predict(X_train))
    val_rmse = root_mean_squared_error(y_val, model.predict(X_valid))
    print(name, "| train RMSE:", round(train_rmse, 2), "| validation RMSE:", round(val_rmse, 2))

In [ ]:
model1 = LinearRegression()
model1.fit(X_tr[['area_sqft']], y_tr)

model2 = LinearRegression()
model2.fit(X_tr[core], y_tr)

model3 = LinearRegression()
model3.fit(X_tr, y_tr)

LinearRegression()

In [ ]:
show("1 feature", model1,  X_tr[['area_sqft']], X_val[['area_sqft']])

1 feature | train RMSE: 15.43 | validation RMSE: 15.04


In [ ]:
show("4 feature2", model2,  X_tr[core], X_val[core])

4 feature | train RMSE: 8.12 | validation RMSE: 8.28


In [ ]:
show("12 features", model3,  X_tr, X_val)

12 features | train RMSE: 7.5 | validation RMSE: 10.27


# Regularisation

In [ ]:
scaler = StandardScaler()
scaler.fit(X_tr)

StandardScaler()

In [ ]:
X_tr_scaled = scaler.transform(X_tr)
X_val_scaled = scaler.transform(X_val)

In [ ]:
lr = LinearRegression()
lr.fit(X_tr_scaled, y_tr)

show("linear regression (raw)", lr, X_tr_scaled, X_val_scaled)

linear regression (raw) | train RMSE: 7.5 | validation RMSE: 10.27


In [ ]:
model = Ridge(alpha=15) # Linear regression + l2 regularisation
model.fit(X_tr_scaled, y_tr)
show("linear regression L2-reg", model, X_tr_scaled, X_val_scaled)

linear regression L2-reg | train RMSE: 9.02 | validation RMSE: 9.13


In [ ]:
pd.DataFrame({"plain": lr.coef_, "ridge_15": model.coef_}, index=X.columns).round(2)

,plain,ridge_15
area_sqft,25.15,20.11
bedrooms,6.27,5.31
age_years,-5.49,-4.66
distance_km,-9.17,-7.73
extra_1,0.20,-0.82
extra_2,-0.11,-1.44
extra_3,1.67,1.58
extra_4,1.41,1.00
extra_5,-1.16,-1.26
extra_6,-1.53,-0.76


In [ ]:
lasso = Lasso(alpha=2) # Linear regression + L1 regularisation
lasso.fit(X_tr_scaled, y_tr)
show("linear regression L2-reg", lasso, X_tr_scaled, X_val_scaled)

linear regression L2-reg | train RMSE: 8.96 | validation RMSE: 7.21


In [ ]:
pd.DataFrame({"lasso_2": lasso.coef_}, index=X.columns).round(2)

,lasso_2
area_sqft,22.88
bedrooms,4.47
age_years,-3.33
distance_km,-7.82
extra_1,-0.00
extra_2,-0.00
extra_3,0.00
extra_4,0.00
extra_5,-0.00
extra_6,-0.00
